# Lab R2 — Chunking: Every Strategy, Scored

**Curriculum §4 · Lab R2**

Hold parser, embedder, retriever and model **fixed**. Vary only the chunker.
Then break the leaderboard down **by document type** — that is where the real lesson is.


## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and pulls the shared `common/` modules.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade above + a restart.


In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())  # fix Colab PIL._typing._Ink mismatch
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu '
                   'rank_bm25 langchain langchain-community langchain-groq '
                   'langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF reportlab pandas'.split())

    # Pull the shared common/ package (config, corpus, golden, obs, scorers, harness).
    REPO = pathlib.Path('/content/genai_practical')
    if not (REPO/'common'/'harness.py').exists():
        # Option A: clone if you've pushed the repo to GitHub — set REPO_URL and uncomment:
        # subprocess.run(['git','clone','$REPO_URL', str(REPO)])
        # Option B: mount Google Drive where you unzipped the repo:
        try:
            from google.colab import drive; drive.mount('/content/drive')
            src = pathlib.Path('/content/drive/MyDrive/genai_practical')
            if src.exists(): REPO = src
        except Exception as e: print('Drive mount skipped:', e)
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e: print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))   # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `common` importable
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| docs in force:', [d['id'] for d in current_docs()])


## 1 · Eight chunking strategies


In [ ]:
import re, numpy as np
from sentence_transformers import SentenceTransformer

def fixed(text, size=512, overlap=0):
    step = max(1, size - overlap); toks = text.split()
    return [' '.join(toks[i:i+size]) for i in range(0, len(toks), step)] or [text]

def by_sentence(text, n=3):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [' '.join(sents[i:i+n]) for i in range(0, len(sents), n)] or [text]

def recursive(text, size=400):
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    return RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=40).split_text(text)

def structure_aware(text):
    parts = re.split(r'(?=Step \d|BTL lending|Enhanced due diligence)', text)
    return [p.strip() for p in parts if p.strip()] or [text]

def semantic(text, thresh=0.55):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(sents) < 3: return [text]
    m = SentenceTransformer(EMBED_MODELS['small'])
    v = m.encode(sents, normalize_embeddings=True)
    out, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        if float(v[i] @ v[i-1]) < thresh: out.append(' '.join(cur)); cur=[sents[i]]
        else: cur.append(sents[i])
    out.append(' '.join(cur)); return out

STRATEGIES = {
  'whole_doc'      : lambda t: [t],
  'fixed_256_o0'   : lambda t: fixed(t, 256, 0),
  'fixed_256_o50'  : lambda t: fixed(t, 256, 50),
  'fixed_512_o0'   : lambda t: fixed(t, 512, 0),
  'sentence_3'     : lambda t: by_sentence(t, 3),
  'recursive_400'  : lambda t: recursive(t, 400),
  'structure'      : structure_aware,
  'semantic'       : semantic,
}
print(len(STRATEGIES),'strategies')


### Contextual chunking
Each chunk is prefixed with its document and version. This is cheap and, on versioned
corpora, often the single most effective change you can make.


In [ ]:
def build_chunks(strategy, contextual=False):
    out=[]
    for d in current_docs():
        for i, piece in enumerate(STRATEGIES[strategy](d['text'])):
            body = f"[{d['doc']} {d['version']}] {piece}" if contextual else piece
            out.append(dict(id=d['id'], doc=d['doc'], version=d['version'],
                            effective=d['effective'], chunk_i=i, text=body))
    return out

for s in STRATEGIES:
    c = build_chunks(s)
    print(f'{s:16s} chunks={len(c):3d}  avg_words={np.mean([len(x["text"].split()) for x in c]):.0f}')


## 2 · Pipeline (only the chunker varies)


In [ ]:
from langchain_groq import ChatGroq
llm   = ChatGroq(model=GEN_MODEL,   temperature=0)
judge = ChatGroq(model=JUDGE_MODEL, temperature=0)
_emb  = {}
def emb(n):
    if n not in _emb: _emb[n]=SentenceTransformer(n)
    return _emb[n]

PROMPT = ("You are Meridian Bank's credit policy copilot. Answer ONLY from context. "
          'Cite the id in square brackets. If absent, reply INSUFFICIENT_CONTEXT.'
          '\n\nContext:\n{ctx}\n\nQuestion: {q}')

@observe(name='rag.request')
def pipeline(query, cfg):
    chunks = build_chunks(cfg['chunk_strategy'], cfg.get('contextual', False))
    m = emb(cfg['embed_model'])
    M = m.encode([c['text'] for c in chunks], normalize_embeddings=True)
    q = m.encode([query], normalize_embeddings=True)[0]
    idx = np.argsort(-(M @ q))[:cfg['k']]
    hits = [dict(chunks[i], score=float((M @ q)[i])) for i in idx]
    ctx  = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    ans  = llm.invoke(PROMPT.format(ctx=ctx, q=query)).content
    span_meta(strategy=cfg['chunk_strategy'], n_chunks=len(chunks),
              returned=[h['id'] for h in hits])
    trace_meta(tags=[cfg['hash']], config=cfg)
    return dict(answer=ans, hits=hits)


## 3 · Run the sweep
9 runs: 8 strategies + the best one repeated with contextual prefixes.


In [ ]:
rows=[]
for s in STRATEGIES:
    cfg = make_config(chunk_strategy=s, embed_model=EMBED_MODELS['base'],
                      k=3, version_filter=True, contextual=False)
    print('running', s)
    rows.append(evaluate(cfg, pipeline, judge=judge))

cfg = make_config(chunk_strategy='recursive_400', embed_model=EMBED_MODELS['base'],
                  k=3, version_filter=True, contextual=True)
rows.append(evaluate(cfg, pipeline, judge=judge))

board = leaderboard(rows, sort_by='hit_at_k')
board[['config','chunk_strategy','hit_at_k','mrr','version_correct','citation','latency_s']]


## 4 · Break it down by document type — the real lesson

A global average hides the finding. Load the per-question detail and group by trap type.


In [ ]:
import glob, pandas as pd
det = pd.concat([pd.read_csv(f).assign(config=f.split('detail_')[1][:-4])
                 for f in glob.glob(str(RESULTS/'detail_*.csv'))])
cfgmap = {r['config']: r.get('chunk_strategy') for r in rows}
det['strategy'] = det.config.map(cfgmap)
det.pivot_table(index='strategy', columns='trap', values='hit_at_k', aggfunc='mean').round(2)


## 5 · What you should conclude

- **`table` questions** favour large / whole-document chunks — splitting a rate table
  separates the LTV from its rate and the answer becomes unrecoverable.
- **`version` questions** improve sharply with **contextual** chunking, because the
  version marker travels with the text instead of living in metadata the retriever ignores.
- **Policy prose** tolerates almost anything.

> **The architectural conclusion:** a single global chunk size is the most common silent
> defect in production RAG. Route chunking by `doctype`, and you now have the table to prove it.

**Next →** `lab03_embeddings.ipynb`
